<a href="https://colab.research.google.com/github/anokhina-rgb/Consecutive-Translation-Trainer/blob/main/Cuba_%D0%A1%D0%BF%D0%BB%D1%96%D1%82%D0%B5%D1%80_%D0%90%D1%83%D0%B4%D1%96%D0%BE_%D0%BD%D0%B0_%D0%A0%D0%B5%D1%87%D0%B5%D0%BD%D0%BD%D1%8F_%D1%82%D0%B0_%D0%93%D0%B5%D0%BD%D0%B5%D1%80%D0%B0%D1%82%D0%BE%D1%80_%D0%9F%D0%B0%D1%83%D0%B7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚖️ Ліцензування та Атрибуція (Обов'язково)

**Розробник Інтеграції (Сплітера):** Анохіна Тетяна, 2025
**Призначення програми:** Обробка аудіо для створення навчальних матеріалів шляхом вставки пауз.

---

### 1. Атрибуція та Авторське Право на Інтеграційний Код

* **Авторське Право:** Анохіна Тетяна © 2025.
* **Ліцензія:** Код інтеграції та логіка сплітування, написана Тетяною Анохіною, розповсюджується на умовах **Creative Commons Attribution 4.0 International (CC BY 4.0)**.
    * **Вимога:** Ви можете вільно ділитися та адаптувати цей код, за умови, що ви **чітко надаєте атрибуцію** (згадку) Тетяні Анохіній.

### 2. Атрибуція Основних Залежностей (Open Source)

Ця програма використовує технології, що мають власні відкриті ліцензії:

#### A. Модель Whisper та whisper-timestamped
Ядро розпізнавання мовлення забезпечується моделлю **OpenAI Whisper** та її модифікацією **`whisper-timestamped`** від linto-ai.

* **Юридична Атрибуція (MIT License):** Вимагається збереження повідомлення про авторські права OpenAI:
    > **Copyright (c) 2022 OpenAI**
    > **Copyright (c) 2022 linto-ai**

#### B. Бібліотека pydub
* **Використання:** Для завантаження та маніпуляції аудіофайлами.
* **Ліцензія:** MIT (Атрибуція зазвичай включена в пакет).

# 📝 Інструкція Користувача та Налаштування

Цей інструмент приймає MP3-файли, транскрибує їх, розділяє аудіо на **повні речення** та **вставляє додаткову паузу** (10 секунд за замовчуванням) після кожного речення.

### Результати обробки:

Після запуску ви отримаєте ZIP-архів, що містить:
1.  **Оброблені MP3-файли:** Аудіо з вставленими паузами.
2.  **Роздатковий матеріал (.txt):** Текстовий файл, де кожне речення позначене як сегмент, з чітким маркером **[PAUSE 10 SEC]**.

### Налаштування:
1.  **`pause_seconds`**: Бажана **довжина паузи** у секундах (за замовчуванням: 10 секунд).
2.  **`model_size`**: Модель Whisper. Зараз встановлено на `"small"` для кращої точності.

### Порядок Дій:
1.  **Запустіть код.**
2.  **Завантажте MP3-файли.**
3.  **Очікуйте** завершення обробки.
4.  **Завантажте ZIP-архів.**

In [ ]:
# Анохіна Тетяна © 2025. Код ліцензовано CC BY 4.0.
# Використовує OpenAI Whisper та whisper-timestamped (MIT License).

# 1️⃣ Install required packages
!pip install -q pydub==0.25.1 torch torchaudio git+https://github.com/linto-ai/whisper-timestamped

# 2️⃣ Import libraries
import os
import re # Added for sentence splitting logic
from pydub import AudioSegment
import whisper_timestamped as whisper
import torch
from google.colab import files
import zipfile

# 3️⃣ User configuration
pause_seconds = 10               # pause length between segments
model_size = "small"             # Model changed to "small" for better segmentation
device = "cuda" if torch.cuda.is_available() else "cpu"
pause_ms = pause_seconds * 1000
PAUSE_MARKER = f"[PAUSE {pause_seconds} SEC]"

# 4️⃣ Temporary folders in Colab
input_folder = "/content/mp3_input"
output_folder = "/content/mp3_output"
os.makedirs(input_folder, exist_ok=True)
os.makedirs(output_folder, exist_ok=True)

# 5️⃣ Upload MP3 files from your computer
print("Upload MP3 files from your computer:")
uploaded = files.upload()
for filename in uploaded.keys():
    os.rename(filename, os.path.join(input_folder, filename))

print(f"\n{len(uploaded)} file(s) uploaded successfully.\n")

# 6️⃣ Load Whisper model
print(f"Loading Whisper model ({model_size}) on device: {device} ...")
model = whisper.load_model(model_size, device=device)
print("Model loaded successfully.\n")

# 7️⃣ Process all MP3 files
files_list = [f for f in os.listdir(input_folder) if f.lower().endswith(".mp3")]
processed_count = 0

for idx, file_name in enumerate(files_list, 1):
    print(f"\nProcessing {file_name} ({idx}/{len(files_list)})...")

    base_name = os.path.splitext(file_name)[0]
    in_path = os.path.join(input_folder, file_name)
    out_audio_path = os.path.join(output_folder, base_name + f"__w_pause_{pause_seconds}s.mp3")
    out_text_path = os.path.join(output_folder, base_name + "__transcript.txt")

    try:
        # Load audio
        original = AudioSegment.from_file(in_path)
        audio = whisper.load_audio(in_path)

        # Transcribe audio (This line is fixed, sentence_split=True is REMOVED)
        result = whisper.transcribe(model, audio, language=None)

        # We rely on the library providing 'sentences' data by default
        sentences = result.get("sentences", [])

        if not sentences:
            print("⚠️ No speech or sentences detected. File skipped.")
            continue

        # Prepare for audio and text output
        silence = AudioSegment.silent(duration=pause_ms)
        final_audio = AudioSegment.empty()
        transcript_content = f"--- Роздатковий Матеріал: {file_name} ---\n\n"

        # Process sentence by sentence
        for i, sentence_data in enumerate(sentences):
            start_ms = int(sentence_data["start"] * 1000)
            end_ms = int(sentence_data["end"] * 1000)

            # Extract the audio chunk for the sentence
            chunk = original[start_ms:end_ms]
            final_audio += chunk

            # Add pause marker to text and pause to audio, except after the last sentence
            if i != len(sentences) - 1:
                final_audio += silence
                transcript_content += f"{sentence_data['text'].strip()} {PAUSE_MARKER}\n\n"
            else:
                transcript_content += f"{sentence_data['text'].strip()}\n\n"


        # 8. Export processed file and transcript
        final_audio.export(out_audio_path, format="mp3")
        with open(out_text_path, "w", encoding="utf-8") as f:
            f.write(transcript_content)

        print(f"✅ Saved audio with pauses: {out_audio_path}")
        print(f"✅ Saved transcript material: {out_text_path}")
        processed_count += 1

    except Exception as e:
        print(f"❌ Error processing {file_name}: {e}")
        # Clean up any potential files for this run
        if os.path.exists(out_audio_path): os.remove(out_audio_path)
        if os.path.exists(out_text_path): os.remove(out_text_path)
        continue

print(f"\n🎉 Processing complete! Successfully processed {processed_count}/{len(files_list)} files.\n")

# 9️⃣ Create a ZIP with all processed files (Audio + Text)
zip_path = "/content/processed_mp3_files.zip"
if processed_count > 0:
    with zipfile.ZipFile(zip_path, 'w') as zipf:
        for f in os.listdir(output_folder):
            zipf.write(os.path.join(output_folder, f), arcname=f)

    # 🔟 Download ZIP to your computer
    print("Downloading all processed files as a single ZIP...")
    files.download(zip_path)
else:
    print("No files were successfully processed. ZIP archive not created.")